In [1]:
from pyspark.sql import functions as F

gold_dim_date = spark.table("gold_dim_date")
gold_dim_product = spark.table("gold_dim_product")
gold_dim_store = spark.table("gold_dim_store")
gold_fact_sales = spark.table("gold_fact_sales")
gold_demand_forecast = spark.table("gold_demand_forecast")

print("Day 6 source tables loaded successfully.")

StatementMeta(, bcd506d6-efcb-4886-aa67-09969493f58b, 3, Finished, Available, Finished, False)

Day 6 source tables loaded successfully.


In [4]:
gold_dim_item = (
    gold_dim_product
    .select("item_id", "dept_id", "cat_id")
    .dropDuplicates(["item_id"])
)

gold_dim_item.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_dim_item")

print("gold_dim_item saved.")

StatementMeta(, bcd506d6-efcb-4886-aa67-09969493f58b, 6, Finished, Available, Finished, False)

gold_dim_item saved.


In [6]:
historical_dates = (
    gold_dim_date
    .select(F.to_date("date").alias("date"))
)

future_dates = (
    gold_demand_forecast
    .select(F.to_date("forecast_date").alias("date"))
)

all_dates = (
    historical_dates
    .unionByName(future_dates)
    .filter(F.col("date").isNotNull())
    .dropDuplicates(["date"])
)

historical_max_date = (
    gold_fact_sales
    .agg(F.max(F.to_date("date")).alias("max_date"))
    .first()["max_date"]
)

gold_dim_reporting_date = (
    all_dates
    .withColumn("year", F.year("date"))
    .withColumn("month_number", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn(
        "is_forecast_date",
        F.col("date") > F.lit(historical_max_date)
    )
    .withColumn(
        "year_month",
        F.date_format("date", "yyyy-MM")
    )
)

gold_dim_reporting_date.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_dim_reporting_date")

print("gold_dim_reporting_date saved.")

StatementMeta(, bcd506d6-efcb-4886-aa67-09969493f58b, 8, Finished, Available, Finished, False)

gold_dim_reporting_date saved.


In [7]:
print("ITEM DIMENSION")
spark.table("gold_dim_item").agg(
    F.count("*").alias("rows"),
    F.countDistinct("item_id").alias("unique_items")
).show()

print("REPORTING DATE DIMENSION")
spark.table("gold_dim_reporting_date").agg(
    F.count("*").alias("rows"),
    F.countDistinct("date").alias("unique_dates"),
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date")
).show()

print("Forecast dates in date dimension:")
spark.table("gold_dim_reporting_date").filter(
    F.col("is_forecast_date")
).count()

StatementMeta(, bcd506d6-efcb-4886-aa67-09969493f58b, 9, Finished, Available, Finished, False)

ITEM DIMENSION
+----+------------+
|rows|unique_items|
+----+------------+
|3049|        3049|
+----+------------+

REPORTING DATE DIMENSION
+----+------------+----------+----------+
|rows|unique_dates|first_date| last_date|
+----+------------+----------+----------+
|1969|        1969|2011-01-29|2016-06-19|
+----+------------+----------+----------+

Forecast dates in date dimension:


56